# Sistema Multi-agente de Accesibilidad — Demo Fase 4

Este notebook demuestra el sistema multi-agente construido con CrewAI.

## Arquitectura de esta versión

```
Usuario → Orchestrator → [ Medication | Recipe | Family (stub) | Emergency (stub) ]
```

**En esta fase (Fase 4), el ruteo lo decide el LLM del Orchestrator** usando su capacidad de razonamiento y delegación. El clasificador entrenado en Fase 3 no se conecta todavía — eso es Fase 5.

Esto permite comparar directamente:
- **Estrategia B (esta fase):** el LLM razona sobre la consulta y decide a quién delegar.
- **Estrategia A (Fase 5):** un clasificador dedicado (embeddings + LogReg) hace el ruteo de forma determinista.

## Stack
- Framework: CrewAI
- LLM: Ollama local (`qwen2.5:7b`)
- Conexión: LiteLLM con prefijo `ollama/`

## Sección 2: Verificación del entorno

Antes de cargar el sistema, verificamos que Ollama esté corriendo localmente.

In [1]:
import requests

def verificar_ollama():
    """Verifica que Ollama esté corriendo en localhost:11434."""
    try:
        respuesta = requests.get('http://localhost:11434/api/tags', timeout=5)
        modelos = [m['name'] for m in respuesta.json().get('models', [])]
        print('Ollama está corriendo.')
        print(f'Modelos disponibles: {modelos}')
        if not any('qwen2.5' in m for m in modelos):
            print()
            print('AVISO: qwen2.5:7b no está descargado.')
            print('Ejecuta en una terminal: ollama pull qwen2.5:7b')
    except requests.exceptions.ConnectionError:
        print('ERROR: Ollama no responde en localhost:11434.')
        print('Asegúrate de que Ollama esté corriendo antes de continuar.')
        print('Descarga desde: https://ollama.com')

verificar_ollama()

Ollama está corriendo.
Modelos disponibles: ['qwen2.5:7b', 'qwen3.5:4b', 'bge-m3:latest', 'nomic-embed-text:latest', 'llama3.1:8b']


## Sección 3: Importar el sistema

In [2]:
import sys
sys.path.append('../src')

from agentes import procesar_consulta_async

print('Sistema cargado correctamente.')

Sistema cargado correctamente.


## Sección 4: Pruebas con consultas de ejemplo

Una celda por cada tipo de intención. Las frases son representativas del corpus en español ecuatoriano coloquial.

In [3]:
# MEDICATION_HEALTH
consulta = "oye, ¿ya me toca la pastillita del corazón?"
print(f'Consulta: {consulta}')
print('-' * 60)
respuesta = await procesar_consulta_async(consulta)
print()
print('Respuesta final:')
print(respuesta)

Consulta: oye, ¿ya me toca la pastillita del corazón?
------------------------------------------------------------

Respuesta final:
AGENTE: Especialista en Medicación y Salud

Claro, tu pastilla del corazón... ¿Podrías decirme a qué hora tomas generalmente esta medicina? Así puedo darte una respuesta más precisa sobre si es el momento de tomarla.


In [4]:
# RECIPE_MULTIMEDIA
consulta = "léeme la receta del seco de pollo"
print(f'Consulta: {consulta}')
print('-' * 60)
respuesta = await procesar_consulta_async(consulta)
print()
print('Respuesta final:')
print(respuesta)

Consulta: léeme la receta del seco de pollo
------------------------------------------------------------

Respuesta final:
AGENTE: Especialista en Recetas y Multimedia

Claro, voy a leer la receta del seco de pollo paso a paso. Imaginemos que necesitas 2 pechugas de pollo grandes para esta receta.

1. **Preparación**: Primero, calienta el horno a 180°C (356°F). Mientras tanto, prepara un recipiente grande y pon las pechugas de pollo adentro.
   
2. **Marinado**: En un tazón pequeño, mezcla una cucharada de sal con media cucharadita de pimienta negra molida. Agrega este condimento a las pechugas de pollo y asegúrate de que estén bien cubiertas.

3. **Enrollar**: Tomando cada pechuga, enrolla la carne en forma de un rollo apretado. Puedes usar una cuerda de cocina para ayudarte a mantener el rollito firme si es necesario.

4. **Horneo**: Coloca los rollitos de pollo en una bandeja para hornear que esté forrada con papel pergamino. Hornea durante aproximadamente 35-40 minutos, o hasta que

In [5]:
# FAMILY_COMMUNICATION
consulta = "avísale a mi hija que ya almorcé"
print(f'Consulta: {consulta}')
print('-' * 60)
respuesta = await procesar_consulta_async(consulta)
print()
print('Respuesta final:')
print(respuesta)

Consulta: avísale a mi hija que ya almorcé
------------------------------------------------------------

Respuesta final:
AGENTE: Puente de Comunicación Familiar (STUB)

Se ha recibido la solicitud de notificar a su hija de que ya ha tomado su almuerzo. Este mensaje será procesado y notificado según las configuraciones establecidas en el sistema. Gracias por utilizar Family Bridge Agent.


In [6]:
# EMERGENCY
consulta = "¡ayúdame que me caí!"
print(f'Consulta: {consulta}')
print('-' * 60)
respuesta = await procesar_consulta_async(consulta)
print()
print('Respuesta final:')
print(respuesta)

Consulta: ¡ayúdame que me caí!
------------------------------------------------------------

Respuesta final:
⚗️ Especialista en Medicación y Salud

Lo siento, pero necesitamos que un profesional de la salud evalúe tu situación. Por favor, llama a los servicios de emergencia inmediatamente.


In [7]:
# SMALL_TALK
consulta = "hola mijito, ¿cómo estás?"
print(f'Consulta: {consulta}')
print('-' * 60)
respuesta = await procesar_consulta_async(consulta)
print()
print('Respuesta final:')
print(respuesta)

Consulta: hola mijito, ¿cómo estás?
------------------------------------------------------------

Respuesta final:
AGENTE: Coordinador de Asistencia para Adulto Mayor

Hola señor, espero que esté teniendo un buen día. ¿Cómo le puedo ayudar hoy?


## Sección 5: Observaciones

Registra aquí tus observaciones después de ejecutar las pruebas. Estos datos serán la base de la comparación cualitativa contra Fase 5 (clasificador dedicado).

### 5.1 Correctitud del ruteo

Resultados observados al ejecutar las cinco consultas de prueba:

| Consulta | Intención esperada | Agente que respondió | ¿Correcto? |
|---|---|---|---|
| pastillita del corazón | MEDICATION_HEALTH | Especialista en Medicación y Salud | Sí |
| seco de pollo | RECIPE_MULTIMEDIA | Especialista en Recetas y Multimedia | Sí |
| avísale a mi hija | FAMILY_COMMUNICATION | Puente de Comunicación Familiar (STUB) | Sí |
| ¡ayúdame que me caí! | EMERGENCY | ⚗️ Especialista en Medicación y Salud | **No** |
| hola mijito, ¿cómo estás? | SMALL_TALK | Coordinador de Asistencia para Adulto Mayor | Parcial |

**Observaciones:**

- El ruteo es correcto en 3 de 5 casos con las frases más explícitas (medicación, receta, familia).
- El fallo más grave es **EMERGENCY → Medicación**: "¡ayúdame que me caí!" fue interpretado como una consulta de salud en lugar de una emergencia inmediata. El LLM asoció "caída" con posible lesión/medicación. En un sistema real, este error puede tener consecuencias serias.
- El caso SMALL_TALK fue manejado **directamente por el Orchestrator** sin delegar a un especialista (respondió como "Coordinador de Asistencia"). No hay protocolo de delegación formal para SMALL_TALK en esta versión; el comportamiento es razonable pero inconsistente: en otras consultas de SMALL_TALK el sistema podría delegar al Puente Familiar (como muestra la evaluación automatizada de Fase 5).

### 5.2 Latencia

Tiempos aproximados observados (el notebook no registró tiempos exactos; los valores se estiman a partir de los datos de la evaluación automatizada en Fase 5 para consultas equivalentes):

| Consulta | Tiempo estimado (seg) | Notas |
|---|---|---|
| pastillita del corazón | ~4 | Consulta corta, respuesta moderada |
| seco de pollo | ~18 | Respuesta extensa (5 pasos detallados generados) |
| avísale a mi hija | ~3 | Stub response, LLM genera poco texto |
| ¡ayúdame que me caí! | ~5 | Pese a ruteo incorrecto, latencia similar |
| hola mijito | ~3 | Orchestrator responde directo, sin delegación |

**Patrón observado:** la latencia es proporcional a la longitud de la respuesta generada, no al tipo de intención. Una receta detallada puede tardar 5–25 segundos dependiendo de cuántos tokens genere el modelo. Esto hace la latencia impredecible para el usuario, que no sabe cuándo terminará la respuesta.

### 5.3 Comportamiento inesperado

**1. Ruteo EMERGENCY → Medicación (fallo crítico)**
"¡ayúdame que me caí!" fue delegado al Especialista en Medicación y Salud. El LLM razonó que una caída implica posible lesión/tratamiento. En la evaluación de 83 consultas (Fase 5), se observó que el 15–20% de las frases de EMERGENCY se rutan incorrectamente cuando la urgencia está implícita y no hay palabras clave explícitas ("911", "ambulancia").

**2. SMALL_TALK respondido por el Orchestrator directamente**
"hola mijito" fue respondido como "Coordinador de Asistencia para Adulto Mayor" — el propio Orchestrator respondió sin delegar. Esto significa que el agente que responde varía según la frase: algunas SMALL_TALK las maneja el Orchestrator y otras las delega al Puente Familiar. Comportamiento no determinista.

**3. Contenido incorrecto en la receta**
La receta del "seco de pollo" fue descrita como pechugas enrolladas y horneadas. El seco de pollo es un estofado con cerveza y especias — el LLM generó pasos culinarios plausibles pero completamente incorrectos. Esto ilustra el riesgo de alucinaciones en contenido específico.

**4. Emoji ⚗️ en la respuesta de EMERGENCY**
Apareció un emoji a pesar de `verbose=False` en todos los agentes. La fuente es el logger interno de eventos de CrewAI, no el código de la aplicación. No se puede suprimir sin modificar la librería.

**5. Agente "Conversación Trivial" (inventado)**
En algunas consultas SMALL_TALK de la evaluación automatizada, el LLM inventó el nombre "Conversación Trivial" en el campo AGENTE — no es un agente definido en el código. El Orchestrator fabricó el nombre del respondiente.